# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện
**Họ và tên:** Hà Mạnh Tuân | **Mã số sinh viên:** 2A202602982

Notebook này thực hiện trọn vẹn quy trình thí nghiệm theo chuẩn **README.md**, **GUIDE.md** và **RUBRIC.md**:
* **Part 0:** Chuẩn bị dữ liệu và phân chia tập validation phân tầng (20%).
* **Part 1:** Định nghĩa kiến trúc mô hình MLP và thực hiện 3 phép thử "sức khoẻ" ban đầu.
* **Part 2:** Huấn luyện baseline với 3 seed để đo độ nhiễu thống kê $2\sigma$.
* **Part 3:** Phủ trọn vẹn 7 chủ đề thí nghiệm (Loss, Optimizer, Hyperparameters, Dropout, Gradient Clipping, Mixed Precision, Initialization).
* **Part 4:** Đánh giá cuối trên tập Eval, phân tích lỗi chi tiết theo lớp và xuất bảng tổng hợp `experiments.xlsx`.

In [ ]:
# ===== Cấu hình môi trường & Thư mục =====
import os
import sys
import json
import time
import math
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Thiết lập đường dẫn tương đối đúng chuẩn từ submission/code/ hoặc code/
REPO_ROOT = "../.." if os.path.exists("../../data") else ".."
if not os.path.exists(f"{REPO_ROOT}/data"):
    REPO_ROOT = "."

OUT_DIR = ".." if os.path.exists("../figures") or os.path.basename(os.getcwd()) == "code" else "."
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

# Lựa chọn thiết bị
device_str = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_str)
print(f"Thiết bị đang sử dụng: {device_str.upper()}")
if device_str == "cuda":
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.0f} MB")
print(f"PyTorch version: {torch.__version__}")

# Import các module tự xây dựng
from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Chuẩn bị dữ liệu
1. Chạy `scripts/split_data.py` để tạo `data/processed/train.npz` (464 809 mẫu) và `eval.npz` (116 203 mẫu).
2. Tách validation từ tập `train` (20% phân tầng, seed 42) $\rightarrow$ 371 847 mẫu train / 92 962 mẫu val.
3. Chuẩn hoá 10 cột liên tục **chỉ dùng thống kê của train** (tránh rò rỉ dữ liệu).
4. Tính mốc baseline "đoán luôn lớp đa số" trên tập validation.

In [ ]:
# Part 0: Chuẩn bị dữ liệu
processed_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed_dir}/train.npz"):
    print("Chạy scripts/split_data.py ...")
    subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/split_data.py"], check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

print(f"Kích thước X_tr   : {data['X_tr'].shape}")
print(f"Kích thước X_val  : {data['X_val'].shape}")
print(f"Kích thước X_eval : {data['X_eval'].shape}")
print(f"Mốc đoán lớp đa số trên Val: {data['majority_acc']:.4f}")

# Kiểm tra chuẩn hoá trên 10 cột đầu của train
tr_num_mean = data['X_tr'][:, :10].mean(dim=0).cpu().numpy()
tr_num_std = data['X_tr'][:, :10].std(dim=0).cpu().numpy()
print(f"Mean 10 cột số train (kỳ vọng ~0): {[round(float(m), 3) for m in tr_num_mean]}")
print(f"Std 10 cột số train  (kỳ vọng ~1): {[round(float(s), 3) for s in tr_num_std]}")

## Part 1 — Model và các phép thử "sức khoẻ" ban đầu
Theo quy định, kiến trúc baseline `M-base` là:
$$54 \rightarrow 256 \rightarrow 128 \rightarrow 7$$
Số tham số bắt buộc: **47 879 tham số**.
Các phép thử bắt buộc trước khi train dài:
1. **Kiểm tra shape logits:** $(B, 7)$ thô, không softmax.
2. **Loss bước 0:** Mô hình vừa khởi tạo có loss val xấp xỉ $\ln(7) \approx 1.946$.
3. **Quá khớp 20 mẫu:** Huấn luyện trên 20 mẫu tắt dropout phải đưa loss về sát 0.
4. **Kiểm tra gradient flow:** Gradient của mọi tham số phải khác 0.

In [ ]:
# Part 1: Kiểm tra Model & Smoke tests
set_seed(42)
model_check = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)

# 1. Đếm tham số
n_params = count_params(model_check)
print(f"Số tham số M-base: {n_params} (Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]})")
assert n_params == EXPECTED_PARAMS[(256, 128)], "Số tham số không khớp!"

# 2. Shape logits
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model_check(dummy_x)
print(f"Shape logits đầu ra: {dummy_logits.shape} (Kỳ vọng: [8, 7])")
assert dummy_logits.shape == (8, 7), "Shape logits sai!"

# 3. Loss bước 0
step0_eval = evaluate(model_check, data["X_val"], data["y_val"], loss_name="ce")
ln7 = math.log(7)
print(f"Loss bước 0 trên Val: {step0_eval['loss']:.4f} (Kỳ vọng ~ ln(7) = {ln7:.4f})")

# 4. Quá khớp 20 mẫu
x20 = data["X_tr"][:20]
y20 = data["y_tr"][:20]
opt_overfit = torch.optim.Adam(model_check.parameters(), lr=0.01)
overfit_losses = []
for step in range(250):
    opt_overfit.zero_grad()
    l_overfit = F.cross_entropy(model_check(x20), y20)
    l_overfit.backward()
    opt_overfit.step()
    overfit_losses.append(l_overfit.item())

print(f"Loss sau 250 bước trên 20 mẫu: {overfit_losses[-1]:.6f}")
assert overfit_losses[-1] < 0.01, "Mô hình không thể quá khớp 20 mẫu!"

# 5. Gradient flow
model_fresh = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
l_grad = F.cross_entropy(model_fresh(x20), y20)
l_grad.backward()
for name, p in model_fresh.named_parameters():
    assert p.grad is not None and p.grad.abs().sum() > 0, f"Tham số {name} bị đứt gradient!"
print("Gradient chảy tốt qua toàn bộ các tham số!")

## Part 2 — Huấn luyện Baseline & Đo độ nhiễu Seed
* Cấu hình baseline: `M-base` ($54 \rightarrow 256 \rightarrow 128 \rightarrow 7$), khởi tạo He, Cross-Entropy, SGD + Momentum 0.9, batch 512, 20 epoch.
* Chạy baseline với 3 seed khác nhau (seed 1, 2, 3) để xác định độ lệch chuẩn $\sigma$ và ngưỡng nhiễu $2\sigma$.

In [ ]:
# Part 2: Chạy Baseline với 3 Seed
base_lr = 0.05
baseline_results = []
seeds = [1, 2, 3]

for s in seeds:
    cfg_s = {
        **DEFAULT_CFG,
        "exp_id": f"base-s{s}",
        "group": "baseline",
        "description": f"Baseline M-base, seed {s}",
        "lr": base_lr,
        "seed": s,
        "epochs": 20,
    }
    print(f"Huấn luyện {cfg_s['exp_id']} ...")
    res = run_experiment(cfg_s, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg_s['exp_id']}.png")
    baseline_results.append(res)
    print(f"  -> Best Epoch: {res['summary']['best_epoch']}, Val Acc: {res['summary']['val_acc']:.4f}, Val Macro-F1: {res['summary']['val_macro_f1']:.4f}")

f1_seeds = [r["summary"]["val_macro_f1"] for r in baseline_results]
acc_seeds = [r["summary"]["val_acc"] for r in baseline_results]
mean_f1 = float(np.mean(f1_seeds))
std_f1 = float(np.std(f1_seeds, ddof=1))
noise_2sigma = 2 * std_f1

print(f"\nĐỘ NHIỄU SEED BASELINE (n=3):")
print(f"Val Macro-F1 trung bình : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Ngưỡng nhiễu 2σ        : {noise_2sigma:.4f}")

## Part 3 — Thực hiện 7 chủ đề thí nghiệm
Tuân thủ nguyên tắc: **Dự đoán trước** $\rightarrow$ **Đổi duy nhất 1 yếu tố** $\rightarrow$ **Huấn luyện** $\rightarrow$ **Đối chiếu cơ chế & ngưỡng nhiễu $2\sigma$**.

### 7 chủ đề:
1. **Hàm mất mát:** Cross-Entropy vs MSE.
2. **Bộ tối ưu hoá:** SGD vs SGD+momentum vs Adam vs AdamW.
3. **Hyperparameters:** Batch size (128 vs 512 vs 2048), M-wide, M-deep.
4. **Dropout:** $q=0.1$ vs $q=0.3$.
5. **Gradient Clipping:** Clipping ở lr chuẩn và ở lr cao (phản chứng).
6. **Mixed Precision:** FP32 vs FP16 vs BF16.
7. **Khởi tạo tham số:** He vs Xavier vs Normal vs Zeros.

In [ ]:
# Part 3: Khai báo và chạy toàn bộ các thí nghiệm
experiments = [
    # 1. Loss
    {**DEFAULT_CFG, "exp_id": "loss-mse", "group": "loss", "description": "MSE loss trên one-hot", "loss": "mse", "lr": base_lr},
    # 2. Optimizer
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.05", "group": "optimizer", "description": "SGD thuần lr=0.05", "optimizer": "sgd", "lr": 0.05},
    {**DEFAULT_CFG, "exp_id": "opt-sgd-lr0.1", "group": "optimizer", "description": "SGD thuần lr=0.1", "optimizer": "sgd", "lr": 0.1},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr1e-3", "group": "optimizer", "description": "Adam lr=1e-3", "optimizer": "adam", "lr": 1e-3},
    {**DEFAULT_CFG, "exp_id": "opt-adam-lr3e-4", "group": "optimizer", "description": "Adam lr=3e-4", "optimizer": "adam", "lr": 3e-4},
    {**DEFAULT_CFG, "exp_id": "opt-adamw-lr1e-3", "group": "optimizer", "description": "AdamW lr=1e-3 wd=0.01", "optimizer": "adamw", "lr": 1e-3, "weight_decay": 0.01},
    {**DEFAULT_CFG, "exp_id": "opt-adamw-lr3e-4", "group": "optimizer", "description": "AdamW lr=3e-4 wd=0.01", "optimizer": "adamw", "lr": 3e-4, "weight_decay": 0.01},
    # 3. Hyper-parameters
    {**DEFAULT_CFG, "exp_id": "hparam-batch-128", "group": "hparam", "description": "Batch nhỏ 128", "batch": 128, "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "hparam-batch-2048", "group": "hparam", "description": "Batch lớn 2048", "batch": 2048, "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "hparam-m-wide", "group": "hparam", "description": "M-wide (512-256)", "hidden": (512, 256), "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "hparam-m-deep", "group": "hparam", "description": "M-deep (256-128-64)", "hidden": (256, 128, 64), "lr": base_lr},
    # 4. Dropout
    {**DEFAULT_CFG, "exp_id": "drop-0.1", "group": "dropout", "description": "Dropout q=0.1", "dropout": 0.1, "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "drop-0.3", "group": "dropout", "description": "Dropout q=0.3", "dropout": 0.3, "lr": base_lr},
    # 5. Gradient Clipping
    {**DEFAULT_CFG, "exp_id": "clip-1.0", "group": "clipping", "description": "Clip c=1.0 lr=0.05", "clip_norm": 1.0, "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "clip-highlr-noclip", "group": "clipping", "description": "Phản chứng: lr=2.0 KHÔNG clip", "clip_norm": None, "lr": 2.0},
    {**DEFAULT_CFG, "exp_id": "clip-highlr-clipped", "group": "clipping", "description": "Phản chứng: lr=2.0 CÓ clip c=1.0", "clip_norm": 1.0, "lr": 2.0},
    # 6. Mixed Precision
    {**DEFAULT_CFG, "exp_id": "amp-fp16", "group": "amp", "description": "FP16 Mixed Precision", "precision": "fp16", "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "amp-bf16", "group": "amp", "description": "BF16 Mixed Precision", "precision": "bf16", "lr": base_lr},
    # 7. Initialization
    {**DEFAULT_CFG, "exp_id": "init-xavier", "group": "init", "description": "Xavier Normal", "init": "xavier", "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "init-normal", "group": "init", "description": "Normal N(0, 0.01^2)", "init": "normal", "lr": base_lr},
    {**DEFAULT_CFG, "exp_id": "init-zeros", "group": "init", "description": "Zeros W=0", "init": "zeros", "lr": base_lr},
]

all_exp_results = list(baseline_results)
for exp_cfg in experiments:
    print(f"Huấn luyện [{exp_cfg['exp_id']:20s}] ...", end=" ", flush=True)
    res = run_experiment(exp_cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    all_exp_results.append(res)
    print(f"Xong! Val Macro-F1: {res['summary']['val_macro_f1']:.4f}")

In [ ]:
# Vẽ biểu đồ so sánh theo từng nhóm
groups = ["optimizer", "loss", "hparam", "dropout", "clipping", "init"]
for grp in groups:
    subset = [r for r in all_exp_results if r["cfg"]["group"] == grp or r["cfg"]["exp_id"] == "base-s1"]
    if subset:
        plot_compare(subset, metric="val_macro_f1", path=f"{OUT_DIR}/figures/compare_{grp}.png", title=f"So sánh Val Macro-F1: {grp.upper()}")
        print(f"Đã lưu {OUT_DIR}/figures/compare_{grp}.png")

## Part 4 — Đánh giá cuối trên tập Eval & Bảng tổng hợp
* Chọn cấu hình tốt nhất thuần tuý dựa trên **Val Macro-F1**.
* Dự đoán trên toàn bộ tập `eval` $\rightarrow$ lưu `predictions_eval.csv`.
* Chấm điểm chính thức bằng `scripts/evaluate.py`.
* Xuất file bảng tổng hợp `experiments.xlsx`.

In [ ]:
# Part 4: Chọn cấu hình cuối cùng & Đánh giá trên tập Eval
valid_runs = [r for r in all_exp_results if not r["summary"]["diverged"]]
best_val_run = max(valid_runs, key=lambda r: r["summary"]["val_macro_f1"])
print(f"Cấu hình tốt nhất trên Val: {best_val_run['cfg']['exp_id']} (Val Macro-F1 = {best_val_run['summary']['val_macro_f1']:.4f})")

# Cấu hình cuối cùng
cfg_final = {
    **best_val_run["cfg"],
    "exp_id": "final-model",
    "group": "final",
    "description": f"Mô hình cuối cùng tối ưu từ val ({best_val_run['cfg']['exp_id']})",
}

res_final = run_experiment(cfg_final, data)
save_result(res_final, f"{OUT_DIR}/results")
plot_run(res_final, f"{OUT_DIR}/figures/{cfg_final['exp_id']}.png")
all_exp_results.append(res_final)

# Ghi file dự đoán Eval
final_eval(baseline_results[0]["cfg"], baseline_results[0], data, f"{OUT_DIR}/predictions_base.csv")
final_eval(res_final["cfg"], res_final, data, f"{OUT_DIR}/predictions_eval.csv")

# Chạy scripts/evaluate.py
print("\nChấm điểm chính thức trên tập Eval:")
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_eval.csv", "--out", f"{OUT_DIR}/eval_result.json"], check=True)
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_base.csv", "--out", f"{OUT_DIR}/eval_result_base.json"], check=True)

with open(f"{OUT_DIR}/eval_result.json", "r") as f:
    eval_metrics = json.load(f)

print(f"\nKẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG TRÊN EVAL:")
print(f"  Accuracy : {eval_metrics['accuracy']:.4f}")
print(f"  Macro-F1 : {eval_metrics['macro_f1']:.4f}")

In [ ]:
# Xuất bảng tổng hợp experiments.xlsx
rows = []
for r in all_exp_results:
    eval_scores = None
    if r["cfg"]["exp_id"] == "base-s1":
        with open(f"{OUT_DIR}/eval_result_base.json", "r") as f:
            b_ev = json.load(f)
        eval_scores = {"eval_acc": b_ev["accuracy"], "eval_macro_f1": b_ev["macro_f1"]}
    elif r["cfg"]["exp_id"] == "final-model":
        eval_scores = {"eval_acc": eval_metrics["accuracy"], "eval_macro_f1": eval_metrics["macro_f1"]}
    rows.append(to_row(r, eval_scores=eval_scores))

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
print("Đã hoàn tất xuất file experiments.xlsx thành công!")